# 🚀 TRAINING RADAR PULSE DEINTERLEAVING - GOOGLE COLAB

**Paper:** [Radar Pulse Deinterleaving with Transformer](https://arxiv.org/abs/2503.13476)

---

## ⚠️ TRƯỚC KHI BẮT ĐẦU:

1. **Enable GPU**: Runtime → Change runtime type → GPU → **T4**
2. **Chạy từng cell theo thứ tự** (Shift+Enter)
3. **Thời gian:** Setup ~1h, Training ~3-12h

---

## 🔧 **BƯỚC 1: Kiểm tra GPU**

In [1]:
# Kiểm tra GPU có sẵn không
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"✅ GPU sẵn sàng!")
else:
    print("\n⚠️  GPU KHÔNG CÓ!")
    print("→ Runtime → Change runtime type → Hardware accelerator → GPU")
    raise RuntimeError("GPU not available")

print(f"{'='*60}")

Tue Mar 17 03:19:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 💾 **BƯỚC 2: Mount Google Drive (Khuyến nghị)**

Lưu checkpoints vào Drive để không mất khi Colab disconnect.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục project
!mkdir -p /content/drive/MyDrive/radar_deinterleaving/data
!mkdir -p /content/drive/MyDrive/radar_deinterleaving/outputs

print("✅ Google Drive đã mount thành công!")

Mounted at /content/drive
✅ Google Drive đã mount thành công!


## 📦 **BƯỚC 3: Clone Repository**

⚠️ **ĐÃ SETUP SẴN:** Username=`MoNguyen127`, Repo=`radar`

📝 **Lưu ý:** Repo phải là **PUBLIC** hoặc dùng Personal Access Token nếu Private.

In [3]:
# ============== THAY ĐỔI ĐÂY ==============
GITHUB_USERNAME = "MoNguyen127"  # ← SỬA DÒNG NÀY!
REPO_NAME = "radar"              # ← Tên repo của bạn
# ============================================

# Xóa repo cũ nếu có
import shutil
import os
if os.path.exists(REPO_NAME):
    print("🗑️  Xóa repo cũ...")
    shutil.rmtree(REPO_NAME)

# Clone repository
print(f"📥 Đang clone repo: {GITHUB_USERNAME}/{REPO_NAME}")
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

# Kiểm tra có thư mục models_implementation không
if os.path.exists(f"{REPO_NAME}/models_implementation"):
    print(f"\n✅ Clone thành công!")
    print(f"📁 Repo location: /content/{REPO_NAME}")
else:
    print("\n❌ LỖI: Không tìm thấy thư mục models_implementation!")
    print("→ Kiểm tra lại username GitHub và tên repo")
    print(f"→ Repo phải là PUBLIC hoặc cần Personal Access Token")
    raise FileNotFoundError("models_implementation not found")

📥 Đang clone repo: MoNguyen127/radar
Cloning into 'radar'...
remote: Enumerating objects: 236, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 236 (delta 98), reused 223 (delta 91), pack-reused 0 (from 0)
Receiving objects: 100% (236/236), 11.30 MiB | 18.52 MiB/s, done.
Resolving deltas: 100% (98/98), done.

✅ Clone thành công!
📁 Repo location: /content/radar


## 📚 **BƯỚC 4: Install Dependencies**

Cài đặt các thư viện cần thiết (~5 phút).

In [4]:
# Install challenge package
print("📦 Cài đặt challenge package...")
%cd /content/{REPO_NAME}
!pip install -e . -q

# Install model dependencies
print("📦 Cài đặt model dependencies...")
%cd models_implementation
!pip install -r requirements.txt -q

print("\n✅ Tất cả dependencies đã được cài đặt!")

📦 Cài đặt challenge package...
/content/radar
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.8 MB/s eta 0:00:00
  Building editable for turing-deinterleaving-challenge (pyproject.toml) ... done
📦 Cài đặt model dependencies...
/content/radar/models_implementation

✅ Tất cả dependencies đã được cài đặt!


## 📥 **BƯỚC 5: Download Dataset từ Google Drive**

⏰ **Thời gian:** ~5-10 phút (giải nén file .rar)

📦 **Dataset:** File .rar đã upload sẵn trên Google Drive

💡 **Tip:** Chỉ cần làm 1 lần, data sẽ lưu vào Drive của bạn!

In [5]:
import os
from pathlib import Path

# ============== ĐỂ FILE .RAR Ở ĐÂU TRONG DRIVE? ==============
# File data.rar đang ở: My Drive/radar_deinterleaving/data.rar
RAR_FILE_PATH = "radar_deinterleaving/data.rar"
# =============================================================

# Chọn nơi lưu data
USE_DRIVE = True  # True = lưu Drive (khuyến nghị), False = Colab local

if USE_DRIVE:
    data_dir = Path('/content/drive/MyDrive/radar_deinterleaving/data')
    print("💾 Lưu vào Google Drive (persistent)...")
else:
    data_dir = Path('/content/data')
    print("💾 Lưu vào Colab local (temporary)...")

data_dir.mkdir(parents=True, exist_ok=True)

# Kiểm tra data đã có chưa (chỉ cần train là đủ)
if (data_dir / 'train').exists():
    print(f"\n✅ Dataset đã tồn tại tại: {data_dir}")
    print("⏭️  Bỏ qua giải nén, dùng data có sẵn.")
else:
    print("\n📥 BƯỚC 1: Cài đặt unrar...")
    !apt-get install -y unrar > /dev/null 2>&1
    print("✅ Đã cài unrar")

    # Đường dẫn file .rar trong Drive đã mount
    rar_source = f'/content/drive/MyDrive/{RAR_FILE_PATH}'

    if not os.path.exists(rar_source):
        print(f"\n❌ LỖI: Không tìm thấy file: {rar_source}")
        print(f"\n📝 HƯỚNG DẪN:")
        print(f"   1. Đảm bảo file data.rar đã upload vào Google Drive")
        print(f"   2. Sửa biến RAR_FILE_PATH ở trên")
        print(f"   3. Ví dụ: Nếu file ở 'My Drive/downloads/data.rar'")
        print(f"      → Sửa thành: RAR_FILE_PATH = 'downloads/data.rar'")
        raise FileNotFoundError(f"File not found: {rar_source}")

    print(f"\n📦 BƯỚC 2: Đang giải nén file .rar...")
    print(f"   Từ: {rar_source}")
    print(f"   Đến: {data_dir.parent}/")
    print("⏰ Thời gian dự kiến: ~3-5 phút\n")

    # Giải nén trực tiếp từ Drive
    !unrar x -o+ "{rar_source}" {data_dir.parent}/

    print("\n✅ Giải nén hoàn tất!")

print(f"\n📊 Data location: {data_dir}")

# Kiểm tra cấu trúc data
if (data_dir / 'train').exists():
    train_files = len(list((data_dir / 'train').glob('*.h5')))
    print(f"   └─ train: {train_files} files")
if (data_dir / 'validation').exists():
    val_files = len(list((data_dir / 'validation').glob('*.h5')))
    print(f"   └─ validation: {val_files} files")
if (data_dir / 'test').exists():
    test_files = len(list((data_dir / 'test').glob('*.h5')))
    print(f"   └─ test: {test_files} files")

💾 Lưu vào Google Drive (persistent)...

✅ Dataset đã tồn tại tại: /content/drive/MyDrive/radar_deinterleaving/data
⏭️  Bỏ qua giải nén, dùng data có sẵn.

📊 Data location: /content/drive/MyDrive/radar_deinterleaving/data
   └─ train: 2500 files
   └─ validation: 250 files
   └─ test: 250 files


In [8]:
%cd /content/radar/models_implementation
!python quick_train.py \
  --data_dir /content/drive/MyDrive/radar_deinterleaving/data \
  --subset_size 100 \
  --epochs 1 \
  --batch_size 2 \
  --window_length 128 \
  --d_model 64 \
  --nhead 4 \
  --num_layers 2 \
  --dim_feedforward 256 \
  --embedding_dim 8

/content/radar/models_implementation
Quick Training Test

Device: cuda

Loading data...
Analyzing files: 100% 2500/2500 [01:35<00:00, 26.23it/s]
Processing files for windows: 100% 2500/2500 [01:37<00:00, 25.69file/s]
Using 100 training samples

Creating model...
Model parameters: 100,872

Training for 1 epochs...
Epoch 1/1: 100% 50/50 [00:01<00:00, 28.53it/s, loss=1.1416, triplets=277855]
Epoch 1 - Avg Loss: 1.0017

Saving model...
Model saved to: /content/radar/models_implementation/quick_test_model.pt

✅ Quick training test completed!

Next steps:
  - Run full training: python train.py --data_dir ../data
  - Test inference: python inference.py --checkpoint quick_test_model.pt


In [7]:
!grep -n "Using .* training samples" /content/radar/models_implementation/quick_train.py
!grep -n "subset_size" /content/radar/models_implementation/quick_train.py

75:    print(f"Using {len(train_subset)} training samples")
42:    subset_size: int = 100,
72:    subset_size = min(subset_size, len(train_dataset))
73:    train_subset = Subset(train_dataset, range(subset_size))
173:    parser.add_argument('--subset_size', type=int, default=100, help='Number of train windows to use')
191:        subset_size=args.subset_size,


## 🧪 **BƯỚC 6: Quick Test (Optional)** - **BỎ QUA**

⚠️ **Bỏ qua bước này!** Quick test cần config thêm, nhảy thẳng sang BƯỚC 7.

In [ ]:
print("⚠️  Bỏ qua quick test - không cần thiết!")
print("✅ Nhảy thẳng sang BƯỚC 7 để config training.")
print("💡 Training chính sẽ tự động validate và báo lỗi nếu có.")

# Quick test cần config data path riêng, không cần thiết cho workflow này
# %cd /content/{REPO_NAME}/models_implementation
# !python quick_train.py

## ⚙️ **BƯỚC 7: Cấu hình Training**

Chọn config phù hợp với thời gian bạn có:

| Config | Epochs | Thời gian (T4) | V-measure |
|--------|--------|----------------|------------|
| Quick | 3 | ~2-3 giờ | ~0.75 |
| Standard | 8 | ~8-12 giờ | ~0.88 |
| Extended | 12 | ~15-20 giờ | ~0.89 |

In [ ]:
# ============== CHỌN CONFIG ==============
TRAINING_CONFIG = "standard"  # Options: "quick", "standard", "extended"
# ==========================================

configs = {
    "quick": {
        "num_epochs": 3,
        "batch_size": 8,
        "validate_every": 1,
        "estimated_hours": "2-3"
    },
    "standard": {
        "num_epochs": 8,
        "batch_size": 8,
        "validate_every": 2,
        "estimated_hours": "8-12"
    },
    "extended": {
        "num_epochs": 12,
        "batch_size": 8,
        "validate_every": 2,
        "estimated_hours": "15-20"
    },
}

config = configs[TRAINING_CONFIG]

print(f"{'='*60}")
print(f"📋 Training Configuration: {TRAINING_CONFIG.upper()}")
print(f"{'='*60}")
print(f"Epochs: {config['num_epochs']}")
print(f"Batch size: {config['batch_size']}")
print(f"Validate every: {config['validate_every']} epochs")
print(f"Thời gian dự kiến: {config['estimated_hours']} giờ")
print(f"{'='*60}")

## 🚀 **BƯỚC 8: BẮT ĐẦU TRAINING**

⏰ **Thời gian:** 3-12 giờ (tùy config)

💡 **Tips:**
- Có thể đóng tab này, training vẫn chạy
- Checkpoints tự động lưu mỗi epoch
- Colab free tier: ~12-15 giờ/session

In [ ]:
import time

%cd /content/{REPO_NAME}/models_implementation

# Output directory
if USE_DRIVE:
    output_dir = "/content/drive/MyDrive/radar_deinterleaving/outputs"
else:
    output_dir = "/content/outputs"

print("\n" + "="*60)
print("🚀 BẮT ĐẦU TRAINING")
print("="*60)
print(f"Data: {data_dir}")
print(f"Output: {output_dir}")
print(f"Epochs: {config['num_epochs']}")
print(f"Thời gian dự kiến: {config['estimated_hours']} giờ")
print("="*60)
print("\n💡 Bạn có thể đi ngủ, để máy chạy qua đêm...\n")

start_time = time.time()

# Training command
!python train.py \
    --data_dir {data_dir} \
    --output_dir {output_dir} \
    --batch_size {config['batch_size']} \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length 1000 \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers 2

elapsed_hours = (time.time() - start_time) / 3600

print("\n" + "="*60)
print(f"🎉 TRAINING HOÀN THÀNH!")
print("="*60)
print(f"⏱️  Thời gian thực tế: {elapsed_hours:.2f} giờ")
print(f"💾 Checkpoints: {output_dir}")
print("="*60)

## 📊 **BƯỚC 9: Tìm Best Model**

In [ ]:
import glob
import os

# Tìm run mới nhất
runs = sorted(glob.glob(f"{output_dir}/run_*"))

if runs:
    latest_run = runs[-1]
    best_model = f"{latest_run}/best_model.pt"

    print(f"{'='*60}")
    print("🏆 BEST MODEL")
    print(f"{'='*60}")

    if os.path.exists(best_model):
        print(f"✅ Model path: {best_model}")
        print(f"📁 Run directory: {latest_run}")

        # Show training log
        log_file = f"{latest_run}/training_log.txt"
        if os.path.exists(log_file):
            print("\n📋 Training Summary (last 10 lines):")
            print("-"*60)
            with open(log_file, 'r') as f:
                lines = f.readlines()
                print("".join(lines[-10:]))
    else:
        print("⚠️  best_model.pt chưa được tạo")
else:
    print("⚠️  Không tìm thấy training runs")

print(f"{'='*60}")

## 📈 **BƯỚC 10: Đánh giá Model**

Evaluate model trên validation set.

In [ ]:
%cd /content/{REPO_NAME}/models_implementation

if os.path.exists(best_model):
    print("📊 Đánh giá model trên validation set...\n")

    !python inference.py \
        --checkpoint {best_model} \
        --data_dir {data_dir} \
        --subset validation \
        --batch_size 8 \
        --save_results {latest_run}/validation_results.json

    print(f"\n✅ Kết quả đã lưu tại: {latest_run}/validation_results.json")
else:
    print("⚠️  Best model không tồn tại. Chạy training trước.")

## 💾 **BƯỚC 11: Download Model về máy**

In [ ]:
from google.colab import files

if runs and os.path.exists(best_model):
    print(f"📥 Đang download: {best_model}")
    files.download(best_model)
    print("✅ Download hoàn tất!")

    # Download validation results nếu có
    results_file = f"{latest_run}/validation_results.json"
    if os.path.exists(results_file):
        print(f"\n📥 Đang download: validation_results.json")
        files.download(results_file)
else:
    print("⚠️  Không có model để download")

## 📊 **BONUS: TensorBoard (Optional)**

Xem training progress với TensorBoard.

In [ ]:
%load_ext tensorboard

# Find tensorboard logs
tb_logs = sorted(glob.glob(f"{output_dir}/run_*/tensorboard"))

if tb_logs:
    latest_tb = tb_logs[-1]
    print(f"📊 Launching TensorBoard: {latest_tb}\n")
    %tensorboard --logdir {latest_tb}
else:
    print("⚠️  Không tìm thấy TensorBoard logs")

---

## 🎯 **KẾT QUẢ MONG ĐỢI**

### Quick Config (3 epochs):
- V-measure: ~0.75
- AMI: ~0.73
- Thời gian: ~2-3 giờ

### Standard Config (8 epochs):
- V-measure: ~0.88
- AMI: ~0.87
- ARI: ~0.81
- Thời gian: ~8-12 giờ

---

## 💡 **TROUBLESHOOTING**

### ❌ "CUDA Out of Memory"
```python
# Giảm batch_size trong BƯỚC 7:
config['batch_size'] = 4  # Thay vì 8
```

### ❌ "Cannot connect to GPU"
- Runtime → Disconnect and delete runtime
- Runtime → Connect
- Chạy lại từ BƯỚC 1

### ❌ Colab disconnect giữa chừng
- Checkpoints đã lưu vào Drive
- Chạy lại từ BƯỚC 2 (Mount Drive)
- Training sẽ tiếp tục từ checkpoint cuối

---

## 📞 **HỖ TRỢ**

- 📖 Hướng dẫn chi tiết: [HUONG_DAN_COLAB.md](https://github.com/YOUR_USERNAME/turing-deinterleaving-challenge/blob/main/models_implementation/HUONG_DAN_COLAB.md)
- 📄 Paper: [arXiv:2503.13476](https://arxiv.org/abs/2503.13476)
- 🐛 Issues: [GitHub Issues](https://github.com/YOUR_USERNAME/turing-deinterleaving-challenge/issues)

---

**🎉 Chúc bạn training thành công!**